[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JanNikolaidovic/python-text-analytics-lab/blob/main/labs/lab08_text_processing/lab08_student.ipynb)

# Lab 08 — Text Processing Toolkit

### Digital Skills and Business Decisioning Using Python

Back in Lab 07 you read a text file, split it into words, and counted how often one word appeared. That was the first real text analytics you did. It also had some rough edges: `"Dress"` and `"dress"` counted as different words, `"dress!"` didn't match `"dress"` at all, and the most "important" words turned out to be `the` and `and`.

This lab fixes all of that. You'll build a small toolkit of text-processing functions, then use it on 600 real customer reviews of women's clothing from an online store. That's the dataset you'll work with for the rest of the course.

In this lab you'll learn to:

* **split** text into words (and on any separator you choose), and **join** words back into text,
* **clean** words: lowercase them and strip punctuation, using the `string` module,
* remove **stopwords** (very common words like `the` and `and`) with a set,
* **count** words the easy way with `Counter`, and find the most common ones,
* **sort** a dictionary by its values with `sorted(..., key=...)`, and
* put it all together into a **word frequency counter** for real reviews, then compare the words in positive and negative reviews.

### Part 1: Splitting and Joining

**`.split()`** breaks a string into a list of smaller strings. Called with no arguments, it splits wherever there is whitespace (spaces, tabs, newlines), however much of it there is.

Run the code cell below:

In [ ]:
review = "Love this dress!  It's sooo pretty and the fit is perfect."

words = review.split()
print(words)
print(len(words))

The double space after `dress!` didn't cause any trouble: with no arguments, `.split()` treats any run of whitespace as a single break. Notice what it *doesn't* do, though: punctuation stays glued to the words (`'dress!'`, `'perfect.'`), and capital letters are untouched (`'Love'`, `"It's"`). We'll fix both in Part 2.

### Splitting on a Separator

Give `.split()` an argument, and it splits on exactly that string instead. This is how you pull apart data where the values are separated by a known character: commas, semicolons, tabs, or anything else.

Run the code cell below:

In [ ]:
sizes_line = "XS,S,M,L,XL"
print(sizes_line.split(","))

record = "5\tLove this dress! It fits perfectly."
rating, text = record.split("\t")
print(rating)
print(text)

The second example is exactly what the review file later in this lab looks like: each line holds a star rating, a **tab** character (written `\t`, like the `\n` newline you already know), and then the review text. Splitting on `"\t"` gives a list of two strings, and **tuple unpacking** from Lab 04 catches them in two variables at once.

Two things to watch:

* `rating` is the **string** `"5"`, not the number `5`. Use `int(rating)` if you want to do maths with it.
* When you give a separator, `.split()` becomes strict: two separators in a row produce an empty string between them. Try it:

In [ ]:
print("a,,b".split(","))
print("love  it".split(" "))
print("love  it".split())

`split(" ")` (with a space) produced an empty string `''` from the double space, while plain `split()` didn't. For splitting text into words, **use `.split()` with no argument**; save the separator version for structured data.

### Joining Words Back Together: `.join()`

**`.join()`** is the mirror image of `.split()`: it glues a list of strings into one string, putting a separator between each pair. The syntax looks backwards at first: the separator comes first, and the list goes inside the parentheses.

Run the code cell below:

In [ ]:
words = ["love", "this", "dress"]

print(" ".join(words))
print(", ".join(words))
print("-".join(words))
print("".join(words))

Read `" ".join(words)` as *"use a space to join these words"*. It's the standard way to rebuild a sentence after you've cleaned the individual words, and you'll use it several times in this lab.

> **`.join()` only works on strings.** `" ".join([1, 2, 3])` raises a `TypeError`. If you have numbers, convert them first, e.g. with a list comprehension: `" ".join([str(n) for n in numbers])`.

### Practice: Splitting and Joining

**Exercise 1: Count the Words in a Title (easy)**

Split the review title below into words and print how many words it has.

*Expected result:* `5`

In [ ]:
title = "Beautiful  but runs   very small"

# TODO: split title into words and print how many there are

In [ ]:
# @solution
title = "Beautiful  but runs   very small"

words = title.split()
print(len(words))

**Exercise 2: Parse a Product Record (medium)**

Each line of a product file stores the fields separated by semicolons: product id, department, and price. Split `line` on `";"`, unpack the three parts into `product_id`, `department`, and `price`, convert `price` to a `float`, and print `f"{department}: {price * 2}"` (the price of two items).

*Expected result:* `Dresses: 297.0`

In [ ]:
line = "1078;Dresses;148.50"

# TODO: split on ";", unpack into three variables, convert price to float, print the department and price * 2

In [ ]:
# @solution
line = "1078;Dresses;148.50"

product_id, department, price = line.split(";")
price = float(price)
print(f"{department}: {price * 2}")

**Exercise 3: Rebuild a Sentence (easy)**

Join the list of words below into a single sentence with spaces between the words, then print it in upper case.

*Expected result:* `RUNS TRUE TO SIZE`

In [ ]:
words = ["runs", "true", "to", "size"]

# TODO: join the words with spaces, then print the result in upper case

In [ ]:
# @solution
words = ["runs", "true", "to", "size"]

sentence = " ".join(words)
print(sentence.upper())

---

### Part 2: Cleaning Words

To a computer, `"Love"`, `"love"`, and `"love!"` are three completely different strings. If you count words without cleaning them first, the same word gets split across several entries and every count comes out wrong. Cleaning (also called **normalizing**) text means making sure the same word always looks the same.

The first step you already know from Lab 02: **lowercase everything**.

In [ ]:
review = "LOVE this dress! Love the color, love the fit."

words = review.lower().split()
print(words)

Better: all three `love`s now look alike. But `love` and `love` are still not the same as the punctuation-covered `dress!` and `color,`. That's the next job.

### Punctuation: the `string` Module

Python's standard library has a module called `string` with a ready-made collection of every punctuation character. You don't need to type them out yourself:

In [ ]:
import string

print(string.punctuation)

Remember `.strip()` from Lab 02? Without arguments it removes whitespace from both ends of a string. Give it a string of characters, and it removes **any of those characters** from both ends instead. Combining it with `string.punctuation` cleans the edges of a word:

Run the code cell below:

In [ ]:
import string

for word in ["dress!", "color,", "(petite)", "fit...", "it's", "t-shirt"]:
    print(f"{word:>10} -> {word.strip(string.punctuation)}")

(The `:>10` inside the f-string just right-aligns each word in a column 10 characters wide, so the output lines up. It doesn't change the word itself.)

`.strip()` only touches the **ends** of a word, so `"it's"` and `"t-shirt"` keep their inner apostrophe and hyphen. That's usually what you want: `it's` and `t-shirt` are real words, and gluing them into `its` and `tshirt` could change their meaning.

### Keeping Only Real Words: `.isalpha()`

After stripping, some "words" end up empty (a lone `"-"` or `"..."` strips down to `""`), and some are numbers like `"4"` or `"5'8"`. String methods starting with `is` answer yes/no questions about a string:

* **`.isalpha()`**: is every character a letter?
* **`.isdigit()`**: is every character a digit?

Run the code cell below:

In [ ]:
for token in ["dress", "5", "5'8", "", "xs", "it's"]:
    print(f"{token!r:>7}  isalpha: {token.isalpha()!s:<5}  isdigit: {token.isdigit()}")

(`!r` shows the string with its quotes, so you can see the empty one; `!s` and `:<5` just line up the columns.)

Notice two things. An empty string gives `False` for both, which makes `.isalpha()` a handy filter for throwing away leftovers. But `"it's"` is also `False` because the apostrophe isn't a letter. Whether you keep words like that is a **choice** you make as the analyst; in this lab we'll keep any word that has at least one letter in it.

### A `tokenize()` Function

Splitting text into a list of clean words is called **tokenizing**, and each word is a **token**. You'll need to do it over and over, so it's the perfect job for a function (Lab 06). Each step is one line you've already seen:

In [ ]:
import string

def tokenize(text):
    """Lowercase text, split it into words, and strip punctuation from each word."""
    tokens = []
    for word in text.lower().split():
        word = word.strip(string.punctuation)
        if word:
            tokens.append(word)
    return tokens


print(tokenize("LOVE this dress! Love the color, love the fit..."))
print(tokenize("Runs small - order a size up!!"))

`if word:` uses **truthiness** from Lab 05: an empty string counts as `False`, so leftovers like the lone `"-"` (which strips down to `""`) are skipped. Every token that comes out is lowercase with clean edges. From here on, whenever you see `tokenize(...)` in this course, this is all it's doing.

### Practice: Cleaning Words

**Exercise 4: Clean One Word (easy)**

Clean the word below so it prints exactly `perfect`: lowercase it and strip punctuation from both ends.

In [ ]:
import string

word = "**PERFECT!!**"

# TODO: lowercase word and strip punctuation from the ends, then print it

In [ ]:
# @solution
import string

word = "**PERFECT!!**"

print(word.lower().strip(string.punctuation))

**Exercise 5: Letters Only (medium)**

Use `tokenize()` on the review below, then use a **list comprehension** (Lab 05) to keep only the tokens where `.isalpha()` is `True`. Print the result.

*Expected result:* `['i', 'am', 'and', 'lbs', 'the', 'xs', 'fits', 'great']`

In [ ]:
review = "I am 5'4 and 120 lbs, the XS fits great."

# TODO: tokenize review, keep only the tokens that are all letters, print the result

In [ ]:
# @solution
review = "I am 5'4 and 120 lbs, the XS fits great."

tokens = tokenize(review)
letters_only = [token for token in tokens if token.isalpha()]
print(letters_only)

`5'4` and `120` are gone because they contain non-letters, while `lbs` survives because `tokenize()` already stripped its trailing comma. Notice what this filter costs you: the customer's height and weight, which could be genuinely useful information about fit. In the next lab you'll learn a tool (regular expressions) that can pull numbers like these *out* of the text instead of throwing them away.

---

### Part 3: Stopwords

Count the words in almost any English text and the top of the list will be `the`, `and`, `a`, `i`, `it`... These are **stopwords**: words so common that they tell you almost nothing about what a particular text is about. A standard cleaning step is to remove them before counting.

A stopword list is a perfect use for a **set** (Lab 04): you only ever ask "is this word in there?", and sets answer that instantly.

In [ ]:
STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "if", "so", "of", "to", "in", "on", "at", "for",
    "with", "as", "by", "from", "up", "out", "about", "into", "than", "too", "very",
    "i", "me", "my", "it", "its", "it's", "this", "that", "these", "those", "is", "am", "are",
    "was", "were", "be", "been", "have", "has", "had", "do", "does", "did", "will", "would",
    "can", "could", "just", "also", "one", "they", "them", "you", "your", "she", "her", "we",
    "i'm", "which", "what", "when", "there", "all", "some", "more", "much", "because", "how",
}

tokens = tokenize("I love the color of this dress and it fits me perfectly")
content_words = [token for token in tokens if token not in STOPWORDS]

print(tokens)
print(content_words)

The name `STOPWORDS` is written in capitals, a Python convention meaning *"this is a constant: set it once, don't change it"*. Python doesn't enforce this; it's a message to whoever reads your code.

Twelve words went in and five came out (`love`, `color`, `dress`, `fits`, `perfectly`), and those five carry nearly all the meaning.

> **Choosing stopwords is an analysis decision, not a fixed rule.** Look at what's *missing* from the list above: `not`, `no`, and `don't`. Many ready-made stopword lists include them, and for counting topics that's fine. But for anything to do with opinions, dropping them is a disaster: `"not comfortable"` becomes `"comfortable"`. Whenever you use someone else's stopword list, read it first.

### Practice: Stopwords

**Exercise 6: Remove the Stopwords (easy)**

Tokenize the review below, remove the stopwords using `STOPWORDS`, and join what's left back into a single string with spaces. Print it.

*Expected result:* `fabric thin see-through returned`

In [ ]:
review = "The fabric was so thin it was see-through, and I returned it."

# TODO: tokenize, drop stopwords, join with spaces, print

In [ ]:
# @solution
review = "The fabric was so thin it was see-through, and I returned it."

tokens = tokenize(review)
kept = [token for token in tokens if token not in STOPWORDS]
print(" ".join(kept))

**Exercise 7: Add Your Own Stopwords (medium)**

In a dataset of clothing reviews, words like `dress` and `top` appear everywhere, so they don't help tell one review apart from another. Make a **new** set called `review_stopwords` containing everything in `STOPWORDS` plus `"dress"`, `"top"`, and `"shirt"` (hint: use `|` with another set). Then print whether `"dress"` is in `STOPWORDS` and whether it's in `review_stopwords`.

*Expected result:* `False` then `True`

In [ ]:
# TODO: build review_stopwords = STOPWORDS plus {"dress", "top", "shirt"} using |
# then print "dress" in STOPWORDS and "dress" in review_stopwords

In [ ]:
# @solution
review_stopwords = STOPWORDS | {"dress", "top", "shirt"}

print("dress" in STOPWORDS)
print("dress" in review_stopwords)

---

### Part 4: Counting With `Counter`

In Lab 05 you counted words by hand with a dictionary and `.get()`:

```python
counts = {}
for word in words:
    counts[word] = counts.get(word, 0) + 1
```

Counting things is so common that the standard library's `collections` module has a tool that does it in one line: **`Counter`**.

Run the code cell below:

In [ ]:
from collections import Counter

words = ["fit", "love", "color", "fit", "small", "love", "fit"]

counts = Counter(words)
print(counts)

A `Counter` is a dictionary under the hood (word → count), so everything you know about dictionaries still works: `counts["fit"]`, `.items()`, `.keys()`, looping. It adds a few extras on top. The most useful one is **`.most_common(n)`**, which returns the `n` most frequent items as a list of `(item, count)` tuples, already sorted from most to least common:

In [ ]:
print(counts.most_common(2))

for word, count in counts.most_common(2):
    print(f"{word}: {count}")

Two more handy differences from a plain dictionary:

In [ ]:
print(counts["fit"])
print(counts["zipper"])

counts.update(["zipper", "fit"])
print(counts["zipper"], counts["fit"])

* Asking a `Counter` for a word it has never seen returns **`0`** instead of raising a `KeyError`. No `.get()` needed.
* **`.update(list)`** counts more items into an existing `Counter`, which is exactly what you want when you process reviews one at a time.

`Counter` works on any list of hashable things, not only words: ratings, department names, sizes, anything you want to tally.

In [ ]:
ratings = [5, 4, 5, 5, 3, 1, 5, 4]
print(Counter(ratings))

### Sorting a Dictionary by Its Values

`.most_common()` sorts a `Counter` for you. But you'll often have a plain dictionary you want sorted by its **values** (say, product prices), and `sorted()` alone doesn't do it: on a dictionary, it sorts the **keys**.

In [ ]:
prices = {"dress": 148, "tank": 38, "jacket": 198, "jeans": 98}

print(sorted(prices))
print(sorted(prices.items()))

`sorted(prices.items())` sorts the `(key, value)` pairs, but a list of tuples is sorted by the **first** element of each tuple, so you still get alphabetical order.

To sort by something else, give `sorted()` a **`key=`** argument: a function that takes one item and returns what to sort by. Here each item is a `(name, price)` tuple, so the function returns position `[1]`:

In [ ]:
def get_price(pair):
    return pair[1]

print(sorted(prices.items(), key=get_price))
print(sorted(prices.items(), key=get_price, reverse=True))

Notice you pass `get_price` **without parentheses**: you're handing `sorted()` the function itself, so that it can call it on every item, rather than calling it once yourself.

Writing a whole `def` for a one-line helper you'll use once feels heavy, so Python has a shorthand called a **`lambda`**: a tiny, unnamed function written in a single line. `lambda pair: pair[1]` means *"a function that takes `pair` and returns `pair[1]`"*, exactly like `get_price`:

In [ ]:
print(sorted(prices.items(), key=lambda pair: pair[1], reverse=True))

Same result. You'll see `key=lambda ...` constantly in other people's code, so it's worth recognizing, but a named function with `def` is always an acceptable alternative.

`key=` works with any function, including built-ins. `key=len` sorts words by their length:

In [ ]:
words = ["fit", "gorgeous", "top", "comfortable", "cute"]
print(sorted(words, key=len))

### Practice: Counting and Sorting

**Exercise 8: Tally the Sizes (easy)**

Use a `Counter` to count how many times each size was ordered, then print the **two** most common sizes with `.most_common()`.

*Expected result:* `[('S', 4), ('M', 3)]`

In [ ]:
from collections import Counter

sizes = ["S", "M", "S", "XS", "L", "M", "S", "M", "S", "XS"]

# TODO: count the sizes with Counter and print the two most common

In [ ]:
# @solution
from collections import Counter

sizes = ["S", "M", "S", "XS", "L", "M", "S", "M", "S", "XS"]

size_counts = Counter(sizes)
print(size_counts.most_common(2))

**Exercise 9: Lowest Stock First (medium)**

Sort the `stock` dictionary by the number of items left, **from fewest to most**, and print each product on its own line as `f"{product}: {count}"`. Use `sorted()` with a `key=`.

*Expected result:*
```
blazer: 2
skirt: 5
sweater: 12
jeans: 30
```

In [ ]:
stock = {"sweater": 12, "jeans": 30, "blazer": 2, "skirt": 5}

# TODO: sort stock.items() by the count (fewest first) and print each product and count

In [ ]:
# @solution
stock = {"sweater": 12, "jeans": 30, "blazer": 2, "skirt": 5}

for product, count in sorted(stock.items(), key=lambda pair: pair[1]):
    print(f"{product}: {count}")

---

### Part 5: Putting It Together, Word Frequencies in Real Reviews

Time for real data. The course repository has a file of **600 customer reviews** from an online women's clothing store (a random sample of a public dataset of about 23,000 reviews that you'll load in full once we get to pandas). Each line holds a star rating from 1 to 5, a tab, and the review text.

The cell below downloads the file into your Colab runtime. You don't need to learn `urllib`; just know that `urlretrieve(url, filename)` saves whatever is at `url` into a local file called `filename`.

In [ ]:
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/JanNikolaidovic/python-text-analytics-lab/main/data/reviews_sample.txt"
urllib.request.urlretrieve(DATA_URL, "reviews_sample.txt")
print("Downloaded reviews_sample.txt")

Open the file browser on the left: `reviews_sample.txt` is there. Now read it with the file skills from Lab 07, and store each review as a dictionary with two keys, `rating` and `text`, inside a list (the "list of dictionaries" shape from Lab 04).

`encoding="utf-8"` tells Python how the characters in the file are stored. It's the standard for text files today; including it avoids strange characters on computers whose default is something else.

In [ ]:
reviews = []

with open("reviews_sample.txt", encoding="utf-8") as file_object:
    for line in file_object:
        rating, text = line.rstrip("\n").split("\t")
        reviews.append({"rating": int(rating), "text": text})

print(f"Loaded {len(reviews)} reviews.")
print(reviews[0])

Before counting words, a quick first look at the ratings. `Counter` works on a list of numbers too, and a list comprehension pulls the ratings out:

In [ ]:
rating_counts = Counter([review["rating"] for review in reviews])

for rating in sorted(rating_counts):
    print(f"{rating} stars: {rating_counts[rating]}")

More than half the reviews give 5 stars and only about 1 in 10 gives 1 or 2. That's very typical of online reviews, and it's a first hint of something you'll analyse properly later: customers who write reviews are, on the whole, happy ones.

### The Most Common Words, Before and After Cleaning

First, the naive approach from Lab 07: split on whitespace and count everything.

In [ ]:
raw_counts = Counter()
for review in reviews:
    raw_counts.update(review["text"].split())

print(raw_counts.most_common(15))

Exactly the problems described at the start of the lab: the top of the list is all stopwords. Further down, the same word is also split across several forms: `The` and `the`, or `dress` and `dress.`, are each counted separately. Now the same count with `tokenize()` and `STOPWORDS`:

In [ ]:
def content_tokens(text):
    """Tokenize text and drop stopwords."""
    return [token for token in tokenize(text) if token not in STOPWORDS]


word_counts = Counter()
for review in reviews:
    word_counts.update(content_tokens(review["text"]))

for word, count in word_counts.most_common(15):
    print(f"{word:<12} {count}")

Now the list reads like a summary of what customers talk about: the garments themselves, how they fit, their size, the fabric, and how much they love them. Notice `not` near the top: it's there because we deliberately left it out of `STOPWORDS`, and in reviews it's doing real work (`not flattering`, `not worth it`).

This is the **word frequency counter** you'll build on for the rest of the course.

Some useful numbers to report about any collection of texts:

In [ ]:
total_tokens = sum(word_counts.values())
vocabulary_size = len(word_counts)
average_length = sum(len(tokenize(review["text"])) for review in reviews) / len(reviews)

print(f"Content words counted: {total_tokens}")
print(f"Distinct content words (vocabulary): {vocabulary_size}")
print(f"Average review length: {average_length:.1f} words")

(`:.1f` formats a number with one digit after the decimal point. And `sum(... for review in reviews)` is a comprehension without the square brackets: `sum()` can add up the values directly, without building a list first.)

### Positive vs. Negative Reviews

Here's a question an analyst would actually ask: **do unhappy customers talk about different things than happy ones?** Split the reviews by rating, count the words in each group separately, and compare.

In [ ]:
positive_counts = Counter()
negative_counts = Counter()

for review in reviews:
    if review["rating"] >= 4:
        positive_counts.update(content_tokens(review["text"]))
    elif review["rating"] <= 2:
        negative_counts.update(content_tokens(review["text"]))

print("Positive (4-5 stars):", positive_counts.most_common(10))
print()
print("Negative (1-2 stars):", negative_counts.most_common(10))

Many words appear in both lists, because both groups are talking about clothes. To see what's **distinctive** about negative reviews, take the top 50 words in each group, turn them into sets, and use the set difference from Lab 04:

In [ ]:
top_positive = {word for word, count in positive_counts.most_common(50)}
top_negative = {word for word, count in negative_counts.most_common(50)}

print("Common in negative reviews only:", sorted(top_negative - top_positive))
print()
print("Common in positive reviews only:", sorted(top_positive - top_negative))

`{word for word, count in ...}` is a **set comprehension**: the same idea as a list comprehension, with curly braces, producing a set.

Read through the two lists. The negative side has `disappointed` and `price`, fit problems (`boxy`, `huge`, `chest`), and a group of words about expectations not matching reality (`online`, `photo`, `looked`, `thought`, `tried`). The positive side is praise and comfort (`perfect`, `flattering`, `comfortable`, `soft`) plus `true` (as in *true to size*). You just did a simple version of what text analytics is all about: turning thousands of words into evidence about *why* customers feel the way they do. Keep in mind that there are only about 60 negative reviews in this sample, so treat these lists as ideas to check on the full dataset, not as conclusions.

### Practice: Real Reviews

**Exercise 10: Average Rating (easy)**

Using the `reviews` list, compute the average star rating across all 600 reviews and print it with two decimal places.

*Expected result:* `Average rating: 4.17`

In [ ]:
# TODO: compute the average of review["rating"] over all reviews and print it with 2 decimals

In [ ]:
# @solution
total = 0
for review in reviews:
    total += review["rating"]

print(f"Average rating: {total / len(reviews):.2f}")

**Exercise 11: A Reusable `top_words()` Function (medium)**

Write a function `top_words(texts, n)` that takes a **list of review texts** and a number `n`, counts the content words across all of them (use `content_tokens()`), and **returns** the `n` most common as a list of `(word, count)` tuples.

Then use it to print the 8 most common words in the **3-star** reviews only.

In [ ]:
# TODO: define top_words(texts, n), then call it on the texts of the 3-star reviews with n=8

In [ ]:
# @solution
def top_words(texts, n):
    counts = Counter()
    for text in texts:
        counts.update(content_tokens(text))
    return counts.most_common(n)


three_star_texts = [review["text"] for review in reviews if review["rating"] == 3]
print(top_words(three_star_texts, 8))

**Exercise 12: How Often Do Reviews Mention Returning? (medium)**

What share of negative reviews (1-2 stars) mention returning the item, compared with positive reviews (4-5 stars)? A review "mentions returning" if any of its tokens is `"return"`, `"returned"`, or `"returning"`.

1. Store those three words in a set called `return_words`.
2. For each group, count the reviews where `set(tokenize(text)) & return_words` is not empty (remember truthiness: an empty set is `False`).
3. Print each share as a percentage with one decimal place.

In [ ]:
# TODO: build return_words, count the reviews in each group that mention returning, print both percentages

In [ ]:
# @solution
return_words = {"return", "returned", "returning"}

negative_texts = [review["text"] for review in reviews if review["rating"] <= 2]
positive_texts = [review["text"] for review in reviews if review["rating"] >= 4]

negative_mentions = 0
for text in negative_texts:
    if set(tokenize(text)) & return_words:
        negative_mentions += 1

positive_mentions = 0
for text in positive_texts:
    if set(tokenize(text)) & return_words:
        positive_mentions += 1

print(f"Negative reviews mentioning returns: {100 * negative_mentions / len(negative_texts):.1f}%")
print(f"Positive reviews mentioning returns: {100 * positive_mentions / len(positive_texts):.1f}%")

---

### Recap: Your Text Processing Toolkit

| Tool | What It Does | Example |
|---|---|---|
| `text.split()` | Split on any whitespace into a list of words | `"a  b".split()` → `['a', 'b']` |
| `text.split(sep)` | Split on an exact separator | `"5\tgreat".split("\t")` → `['5', 'great']` |
| `sep.join(list)` | Glue a list of strings together with `sep` | `" ".join(["a", "b"])` → `'a b'` |
| `import string` | Standard module of useful character collections | `string.punctuation` |
| `word.strip(chars)` | Remove any of `chars` from both ends | `"fit!".strip(string.punctuation)` → `'fit'` |
| `.isalpha()` / `.isdigit()` | Is every character a letter / a digit? | `"xs".isalpha()` → `True` |
| `tokenize(text)` | Our function: lowercase, split, strip punctuation | `tokenize("Love it!")` → `['love', 'it']` |
| Stopword set | Common words to drop, checked with `in` | `[w for w in tokens if w not in STOPWORDS]` |
| `Counter(list)` | Count every item in a list | `Counter(["a", "b", "a"])` |
| `.most_common(n)` | The `n` most frequent items, as `(item, count)` tuples | `counts.most_common(10)` |
| `.update(list)` | Add more items to an existing `Counter` | `counts.update(tokens)` |
| `sorted(..., key=f)` | Sort by whatever `f` returns for each item | `sorted(d.items(), key=lambda p: p[1])` |
| `lambda x: ...` | A tiny unnamed one-line function | `lambda pair: pair[1]` |
| `{x for x in ...}` | Set comprehension | `{w for w, c in counts.most_common(50)}` |

The whole lab in one sentence: **clean text is lowercase, split into words, stripped of punctuation and stopwords, and once it's clean, `Counter` turns any pile of reviews into a ranked list of what people are talking about.**